# v74 Phase G — GRフリー dip場エンジン単体 提出ノート

- **GR不使用・外部モデルファイル不要の完全自己完結**。train/ から場(セグメントdipドナー)を構築し、
  κ係数(α/β)はローカルと同一の GroupKFold(5) を train 内で再現して **5セットをフィット → テストは5セット全部で推論して平均**。
- エンジン = ローカル `phaseG_struct2_v74.py` v3 lean構成(SEG_LEN=0 / VEC=1 / QW=0 / ROBUST=0 + perp係数リッジ + ドリフト±150ftクリップ)。
  ローカルCV(fold-honest, 773井, overall pooled)= **9.832**。
- 予測式: `TVT̂ = anchor + clip(cumsum(α_bin·(−ΔZ) + β_bin·(g·ΔXY)), ±150)`(anchor=最終既知TVT_input)。

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd

t00 = time.time()
# ---- パス自動判定(Kaggle / ローカル) ----
KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    ROOT = glob.glob('/kaggle/input/*rogii*')[0]
else:
    ROOT = r'c:/Users/kosaka256/Documents/rogii_claude/rogii-wellbore-geology-prediction'
TRAIN_DIR = os.path.join(ROOT, 'train')
TEST_DIR = os.path.join(ROOT, 'test')
SAMPLE = os.path.join(ROOT, 'sample_submission.csv')

# ---- エンジン定数(ローカル phaseG v3 lean = LB10.938 の正構成) ----
PGATE = 0.35
THETA0 = 118.4
KNN_K, KNN_H = 15, 500.0
KBINS = [0.0, 750.0, 1500.0, 2500.0, 4000.0, 1e18]
KAPPA_REGIMES = [0.0, 1000.0, 1500.0, 2000.0]
NBIN = len(KBINS) - 1
CMAX = 0.30
K_FIX = 16
CLIP_DRIFT = 150.0
N_FOLDS = 5
_C0, _S0 = np.cos(np.radians(THETA0)), np.sin(np.radians(THETA0))
print('ROOT =', ROOT)

In [ ]:
# ================= エンジン(phaseG v3 lean の移植) =================
def load_well(fp, need_tvt):
    cols = ['X', 'Y', 'Z', 'TVT', 'TVT_input'] if need_tvt else ['X', 'Y', 'Z', 'TVT_input']
    w = pd.read_csv(fp, usecols=cols)
    ti = w['TVT_input'].to_numpy(float)
    fin = np.where(np.isfinite(ti))[0]
    if len(fin) == 0:
        return None
    s = int(fin.max())
    X, Y, Z = w['X'].to_numpy(float), w['Y'].to_numpy(float), w['Z'].to_numpy(float)
    n = len(X) - 1 - s
    if n < 2:
        return dict(s=s, n=0, tin=ti)   # 評価行ほぼ無し → anchorホールドで処理
    dX = np.diff(X)[s:]; dY = np.diff(Y)[s:]
    ndz = -np.diff(Z)[s:]
    Lxy = np.maximum(np.hypot(dX, dY), 1e-3)
    p_row = (dX * _C0 + dY * _S0) / Lxy
    d = dict(s=s, n=n, dX=dX, dY=dY, ndz=ndz, Lxy=Lxy, arc=np.cumsum(Lxy),
             Xl=X[s:], Yl=Y[s:], anchor=float(ti[s]), tin=ti,
             perp=(np.abs(p_row) < PGATE))
    if need_tvt:
        tvt = w['TVT'].to_numpy(float)
        d['R0'] = tvt[s + 1:] - tvt[s]
    return d

def segment_well(wd, with_slopes):
    """弧長基底ランプフィット。with_slopes=True(train)は c/se も返す。test はセグメント幾何のみ。"""
    n, A = wd['n'], wd['arc']
    K = min(K_FIX, max(1, n // 4))
    total = float(A[-1])
    edges = np.linspace(0.0, total, K + 1)
    segid = np.clip(np.searchsorted(edges[1:], A, side='left'), 0, K - 1)
    mid = np.empty((K, 2)); az = np.empty(K)
    Xl, Yl = wd['Xl'], wd['Yl']
    for j in range(K):
        rows = np.where(segid == j)[0]
        j0, j1 = (int(rows[0]), int(rows[-1])) if len(rows) else (0, n - 1)
        az[j] = np.arctan2(Yl[j1 + 1] - Yl[j0], Xl[j1 + 1] - Xl[j0])
        mid[j] = ((Xl[j0] + Xl[j1 + 1]) / 2.0, (Yl[j0] + Yl[j1 + 1]) / 2.0)
    if not with_slopes:
        return segid, mid, az, None, None
    phi = np.column_stack([np.clip(A - edges[j], 0.0, edges[j + 1] - edges[j]) for j in range(K)])
    y = wd['R0'] - np.cumsum(wd['ndz'])
    m = np.isfinite(y)
    if m.sum() < K + 2:
        return segid, mid, az, None, None
    c, res, rank, _ = np.linalg.lstsq(phi[m], y[m], rcond=None)
    dof = max(int(m.sum()) - K, 1)
    ssr = float(res[0]) if len(res) else float(np.sum((phi[m] @ c - y[m]) ** 2))
    cov = (ssr / dof) * np.linalg.pinv(phi[m].T @ phi[m])
    se = np.sqrt(np.maximum(np.diag(cov), 1e-12))
    return segid, mid, az, c, se

def build_field(train_wells):
    px, py, pc, paz, pwi = [], [], [], [], []
    for wd in train_wells:
        if wd.get('seg') is None or wd['seg'][3] is None:
            continue
        segid, mid, az, c, se = wd['seg']
        ok = np.isfinite(c) & (np.abs(c) <= CMAX) & np.isfinite(se)
        px.append(mid[ok, 0]); py.append(mid[ok, 1]); pc.append(c[ok]); paz.append(az[ok])
        pwi.append(np.full(int(ok.sum()), wd['wi'], float))
    F = dict(x=np.concatenate(px), y=np.concatenate(py), c=np.concatenate(pc),
             az=np.concatenate(paz), wi=np.concatenate(pwi))
    F['qw'] = np.ones(len(F['x']))                        # lean: QW=0
    F['ux'], F['uy'] = np.cos(F['az']), np.sin(F['az'])
    F['p0'] = F['ux'] * _C0 + F['uy'] * _S0
    return F

def query_well(F, mids, own, mdbuf=0.0):
    """lean VEC=1: θ0拘束回帰のみ(ROBUST=0)。own=-1 でLOO無効(test)。"""
    keep = F['wi'] != own
    fx, fy = F['x'][keep], F['y'][keep]
    fc, fqw = F['c'][keep], F['qw'][keep]
    fp0 = F['p0'][keep]
    dx = fx[None, :] - mids[:, 0:1]; dy = fy[None, :] - mids[:, 1:2]
    d2 = dx * dx + dy * dy
    if mdbuf > 0:
        d2 = np.where(d2 >= mdbuf * mdbuf, d2, np.inf)
    kk = min(KNN_K, d2.shape[1] - 1) if d2.shape[1] > 1 else 1
    idx = np.argpartition(d2, kk - 1, axis=1)[:, :kk]
    r = np.arange(len(mids))[:, None]
    d2s = d2[r, idx]
    c_s, p0_s = fc[idx], fp0[idx]
    w = np.exp(np.maximum(-d2s / (2 * KNN_H ** 2), -700.0)) * fqw[idx]
    w[~np.isfinite(d2s)] = 0.0
    dead = w.sum(1) <= 1e-300
    with np.errstate(invalid='ignore'):
        dd = np.sqrt(np.nanmedian(np.where(np.isfinite(d2s), d2s, np.nan), axis=1))
    dd[~np.isfinite(dd)] = KBINS[-2] * 2
    s_hat = (w * c_s * p0_s).sum(1) / ((w * p0_s ** 2).sum(1) + 1e-9)
    g = np.column_stack([s_hat * _C0, s_hat * _S0])
    g[dead] = 0.0
    gn = np.hypot(g[:, 0], g[:, 1])
    big = gn > CMAX
    if big.any():
        g[big] *= (CMAX / gn[big])[:, None]
    return g, dd

def design_blocks(F, wd, mdbuf):
    """κ正規方程式用の設計行列ブロック(井戸×レジーム)。"""
    g, dd = query_well(F, wd['seg'][1], wd['wi'], mdbuf=mdbuf)
    sid = wd['seg'][0]
    f = g[sid, 0] * wd['dX'] + g[sid, 1] * wd['dY']
    a = wd['ndz']
    par = ~wd['perp']
    b_row = np.digitize(dd, KBINS[1:-1])[sid]
    cols = []
    for b in range(NBIN):
        m = (b_row == b) & par
        cols.append(np.cumsum(np.where(m, a, 0.0)))
        cols.append(np.cumsum(np.where(m, f, 0.0)))
    cols.append(np.cumsum(np.where(wd['perp'], a, 0.0)))
    cols.append(np.cumsum(np.where(wd['perp'], f, 0.0)))
    G = np.column_stack(cols)
    ok = np.isfinite(wd['R0'])
    G = np.nan_to_num(G[ok]); y = wd['R0'][ok]
    return G.T @ G, G.T @ y

def solve_kappa(A, yv):
    npar = 2 * NBIN
    ncol = npar + 2
    A = A.copy()
    lam_p = 0.02 * float(np.trace(A)) / ncol              # perp係数リッジ(v3の根治策)
    A[npar, npar] += lam_p; A[npar + 1, npar + 1] += lam_p
    coef = np.linalg.lstsq(A, yv, rcond=None)[0]
    return dict(alpha=np.clip(coef[0:npar:2], -0.25, 1.5),
                beta=np.clip(coef[1:npar:2], -0.25, 1.5),
                ap=float(np.clip(coef[npar], -0.25, 1.5)),
                bp=float(np.clip(coef[npar + 1], -0.25, 1.5)))

def predict(F, wd, K, own=-1):
    g, dd = query_well(F, wd['seg'][1], own, mdbuf=0.0)
    sid = wd['seg'][0]
    f = g[sid, 0] * wd['dX'] + g[sid, 1] * wd['dY']
    b_row = np.digitize(dd, KBINS[1:-1])[sid]
    av = K['alpha'][b_row]; bv = K['beta'][b_row]
    av = np.where(wd['perp'], K['ap'], av)
    bv = np.where(wd['perp'], K['bp'], bv)
    resid = np.clip(np.cumsum(av * wd['ndz'] + bv * f), -CLIP_DRIFT, CLIP_DRIFT)
    return wd['anchor'] + resid                            # rows s+1 .. end
print('engine defined')

In [ ]:
# ================= train ロード → 場構築 → κ 5-fold フィット =================
wids = sorted(set(os.path.basename(f).split('__')[0]
                  for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))))
train_wells = []
for wi, w in enumerate(wids):
    wd = load_well(os.path.join(TRAIN_DIR, f'{w}__horizontal_well.csv'), need_tvt=True)
    if wd is None or wd['n'] < 2:
        continue
    wd['wi'] = wi; wd['wid'] = w
    wd['seg'] = segment_well(wd, with_slopes=True)
    train_wells.append(wd)
print(f'train wells={len(train_wells)}  ({time.time()-t00:.0f}s)')
F = build_field(train_wells)
print(f'field donors={len(F["x"])}')

# 設計ブロックを井戸ごとに一度だけ計算(レジーム4種を合算)→ fold別に加算するだけ
blocks = {}
for wd in train_wells:
    if wd['seg'][3] is None or 'R0' not in wd:
        continue
    Aw = 0; yw = 0
    for R in KAPPA_REGIMES:
        Ab, yb = design_blocks(F, wd, R)
        Aw = Aw + Ab; yw = yw + yb
    blocks[wd['wi']] = (Aw, yw)
print(f'design blocks={len(blocks)}  ({time.time()-t00:.0f}s)')

# ローカルと同一の GroupKFold(5)(=sorted wid 順のインデックスを5分割)
from sklearn.model_selection import GroupKFold
gi = np.arange(len(train_wells))
folds = list(GroupKFold(N_FOLDS).split(gi, gi, gi))
K_sets = []
for fold, (tr_i, va_i) in enumerate(folds):
    A = 0; yv = 0
    for i in tr_i:
        wi = train_wells[i]['wi']
        if wi in blocks:
            A = A + blocks[wi][0]; yv = yv + blocks[wi][1]
    K = solve_kappa(A, yv)
    K_sets.append(K)
    print(f'fold{fold}: alpha={np.round(K["alpha"],3)} beta={np.round(K["beta"],3)} '
          f'ap={K["ap"]:.3f} bp={K["bp"]:.3f}')

In [ ]:
# ================= test 推論(5セット平均)+ ★共変量センサス → submission =================
test_ids = sorted(set(os.path.basename(f).split('__')[0]
                      for f in glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv'))))
print(f'test wells={len(test_ids)}')
pred_map = {}
census = []            # (median_dd, n_eval_rows, prefix_rows, arc_len)
nf = 0
for w in test_ids:
    wd = load_well(os.path.join(TEST_DIR, f'{w}__horizontal_well.csv'), need_tvt=False)
    if wd is None:
        continue
    tin = wd['tin']
    if wd['n'] < 2:
        s = wd['s']
        for ri in np.where(~np.isfinite(tin))[0]:
            pred_map[f'{w}_{ri}'] = float(tin[s])
        nf += 1
        continue
    wd['wi'] = -1
    wd['seg'] = segment_well(wd, with_slopes=False)
    _, dd_seg = query_well(F, wd['seg'][1], -1)
    census.append((float(np.median(dd_seg)), int(np.sum(~np.isfinite(tin))),
                   int(np.sum(np.isfinite(tin))), float(wd['arc'][-1])))
    preds = np.mean([predict(F, wd, K, own=-1) for K in K_sets], axis=0)
    s = wd['s']
    for k, ri in enumerate(range(s + 1, s + 1 + wd['n'])):
        if not np.isfinite(tin[ri]):
            pred_map[f'{w}_{ri}'] = float(preds[k])
    for ri in np.where(~np.isfinite(tin[:s + 1]))[0]:
        j = np.where(np.isfinite(tin[:ri + 1]))[0]
        pred_map[f'{w}_{ri}'] = float(tin[j[-1]]) if len(j) else float(wd['anchor'])
print(f'pred rows={len(pred_map)}  fallback wells={nf}  ({time.time()-t00:.0f}s)')

# ---- ★共変量センサス(隠しtestの構成をログに出す。ローカル訓練セットと同じビン) ----
cz = np.array(census)
bins = [0, 300, 600, 900, 1200, 1800, 3000, 1e18]
print('==== TEST COVARIATE CENSUS (median donor distance) ====')
for a, b in zip(bins[:-1], bins[1:]):
    m = (cz[:, 0] >= a) & (cz[:, 0] < b)
    wsum = cz[m, 1].sum() / max(cz[:, 1].sum(), 1) * 100
    print(f'  dd {a:>5.0f}-{b if b < 1e18 else 9999:>5.0f}ft: wells={int(m.sum()):4d}  row_weight={wsum:5.1f}%')
print(f'  dd median={np.median(cz[:,0]):.0f}ft  mean={np.mean(cz[:,0]):.0f}ft')
print(f'  eval_rows median={np.median(cz[:,1]):.0f}  prefix_rows median={np.median(cz[:,2]):.0f}  '
      f'arc median={np.median(cz[:,3]):.0f}ft')
print('==== END CENSUS ====')

sub = pd.read_csv(SAMPLE)
sub['tvt'] = [pred_map.get(i, np.nan) for i in sub['id']]
miss = sub['tvt'].isna()
if miss.any():
    wells_of = sub['id'].str.rsplit('_', n=1).str[0]
    med = sub.groupby(wells_of)['tvt'].transform('median')
    sub.loc[miss, 'tvt'] = med[miss]
    sub['tvt'] = sub['tvt'].fillna(0.0)
    print(f'WARN: filled {int(miss.sum())} missing rows')
sub.to_csv('submission.csv', index=False)
print('submission.csv saved:', len(sub), f'rows  ({time.time()-t00:.0f}s)')
sub.head()